## 개요

`category_topic_dynamic_rules_notebook.py`는  
기존처럼 `clue_keywords`, `generic_terms`, `hard rules`를 코드에 하드코딩하지 않고,  
**카테고리별 리뷰 샘플을 바탕으로 LLM이 규칙을 먼저 생성한 뒤**  
그 규칙을 이용해 동일한 구조의 topic tagging을 수행하는 노트북입니다.

---

## 이 노트북이 해결하는 문제

기존 방식은 아래 한계가 있었습니다.

- `리모컨 사용성`에 맞춘 키워드를 코드에 직접 넣어야 함
- `화질`, `음질`, `연결성` 등 다른 카테고리로 확장할 때 다시 수정 필요
- `전반적 긍정 / 전반적 부정`의 범위를 카테고리별로 다르게 관리하기 어려움

이 노트북은 이를 개선해서:

- `카테고리별 샘플 메모`를 읽고
- LLM이 해당 카테고리에 맞는
  - `clue_keywords`
  - `generic_terms`
  - `overall usage rule`
  - `specific reason rule`
  - `non_overall_examples`
  를 자동 생성하고
- 이 값을 테이블로 저장한 뒤
- 이후 topic pool 생성과 최종 분류에 그대로 재사용합니다.

---

## 처리 흐름

### 1. 대상 메모 샘플 로드
아래 기준으로 샘플 메모를 불러옵니다.

- `cate_1_depth`
- `cate_2_depth`
- `sc_measurement`

현재 설정에서는:

- `07. 스마트 사용성`
- `07-06. 리모컨 사용성`

을 기준으로 작동합니다.

또한:

- `TARGET_SC_MEASUREMENT = 1` → 현재 분류 대상은 긍정
- `RULE_SC_MEASUREMENTS = [1, -1]` → 규칙 프로파일은 긍정/부정 둘 다 생성

---

### 2. 규칙 프로파일 생성
LLM이 각 `sc_measurement`별로 아래 값을 생성합니다.

- `overall_topic_label`
  - 예: `전반적 긍정`, `전반적 부정`
- `clue_keywords`
  - 특정 이유가 있는 리뷰를 식별하는 단서
- `generic_terms`
  - 아주 일반적인 감성 표현
- `overall_usage_rule`
  - 전반적 긍/부정 사용 조건
- `specific_reason_rule`
  - 구체적 이유가 있는 리뷰는 세부 주제로 보내는 규칙
- `non_overall_examples`
  - 전반적 긍/부정으로 가면 안 되는 예시 표현

이 결과는 `RULE_PROFILE_TABLE`에 저장됩니다.

즉, 하드코드 대신 **카테고리별 동적 규칙 테이블**을 먼저 만드는 방식입니다.

---

### 3. 규칙 프로파일 로드
분류 대상 polarity에 맞는 규칙을 불러옵니다.

예:
- 현재 `TARGET_SC_MEASUREMENT = 1`이면
- 긍정용 `clue_keywords`, `generic_terms`, `overall_usage_rule` 등을 읽어옵니다.

이 값들이 이후 단계의 prompt와 후처리에 사용됩니다.

---

### 4. Topic Pool 생성
LLM은 샘플 메모와 함께,
앞 단계에서 생성한 규칙 프로파일을 참조하여
최대 `17개`의 topic pool을 생성합니다.

구성 원칙:

- 최소 7개, 최대 17개 주제
- `전반적 긍정` 또는 `전반적 부정`은 반드시 포함
- 다만 그 라벨은 **아주 짧고 이유 없는 문장**에만 사용
- 나머지는 모두 구체적인 이유 기반 주제
- 주제명은 한국어 기준으로 생성
- 설명도 한국어로 간단히 생성

이 결과는 `TOPIC_POOL_TABLE`에 저장됩니다.

---

### 5. Topic Classification
각 memo를 topic pool 중 하나로 분류합니다.

분류 시에도 다음 값을 다시 사용합니다.

- `clue_keywords`
- `generic_terms`
- `overall_usage_rule`
- `specific_reason_rule`
- `non_overall_examples`

즉, topic pool 생성 때와 같은 기준으로
최종 분류 단계도 수행합니다.

---

### 6. 전반적 긍/부정 재검토
모델이 어떤 memo를 `전반적 긍정` 또는 `전반적 부정`으로 분류했더라도,
후처리에서 한 번 더 확인합니다.

판정 방식:

- memo 안에 `clue_keywords`가 있으면
- 그 문장은 구체적인 이유가 있는 것으로 보고
- `전반적 긍정/부정`으로 남기지 않음

이 경우:
- `전반적 긍정/부정`을 제외한 주제들만 가지고
- 다시 한 번 재분류합니다.

이렇게 해서
`전반적 긍정/부정`의 범위를 최대한 좁게 유지합니다.

---

### 7. 희소 주제 병합
최종 분류 후 topic별 리뷰 수를 계산합니다.

규칙:

- 하나의 topic에 최소 `10개` 이상 리뷰가 있어야 함
- `10개 미만` topic은 독립 주제로 유지하지 않음
- 대신 `기타(...)` 형태로 병합

예:
- `기타(버튼 반응이 세밀함, 특정 앱 조작이 쉬움, 페어링이 빠름)`

이렇게 해서 topic 수가 지나치게 잘게 쪼개지지 않도록 관리합니다.

---

## 최종 산출물

### 1. `RULE_PROFILE_TABLE`
카테고리별 동적 규칙 프로파일 저장 테이블

포함 예시:
- `overall_topic_label`
- `clue_keywords_json`
- `generic_terms_json`
- `overall_usage_rule`
- `specific_reason_rule`
- `non_overall_examples_json`

---

### 2. `TOPIC_POOL_TABLE`
생성된 주제풀 테이블

포함 예시:
- `topic_order`
- `topic`
- `description`
- `representative_memos_json`

---

### 3. `DETAIL_TABLE`
메모별 분류 결과

포함 예시:
- `memo`
- `topic`
- `description`

---

### 4. `SUMMARY_TABLE`
주제별 집계 결과

포함 예시:
- `topic`
- `review_count`
- `review_share`
- `review_share_pct`

---

## 이 방식의 장점

### 카테고리 확장성
리모컨 사용성뿐 아니라
향후 `화질`, `음질`, `연결성`, `설치`, `가격` 등으로 확장할 때도
코드 하드코딩 없이 같은 구조를 적용할 수 있습니다.

### 전반적 긍/부정 정밀화
카테고리별로 `전반적 긍/부정`을 얼마나 좁게 볼지
리뷰 데이터를 보고 동적으로 설정할 수 있습니다.

### 규칙 재사용 가능
한 번 만든 `RULE_PROFILE_TABLE`을 저장해두면,
이후 같은 카테고리에서 반복 실행할 때 같은 기준을 재사용할 수 있습니다.

### 사람 검토 가능성
규칙 자체가 테이블로 남기 때문에,
사람이 확인하고 수정하기도 쉽습니다.

---

## 현재 테스트 설정

현재 노트북은 아래 기준으로 테스트됩니다.

- `cate_1_depth = 07. 스마트 사용성`
- `cate_2_depth = 07-06. 리모컨 사용성`
- `TARGET_SC_MEASUREMENT = 1`

즉 현재는 **리모컨 사용성 긍정 리뷰**를 기준으로 topic pool과 분류를 수행합니다.

다만 규칙 프로파일은:
- `1`
- `-1`

두 polarity 모두 생성되므로,
이후 부정 버전으로도 확장 가능합니다.

---

## 한 줄 요약

이 노트북은  
**카테고리별 리뷰 샘플을 바탕으로 LLM이 동적 분류 규칙을 먼저 생성하고,  
그 규칙을 기반으로 topic pool 생성과 최종 분류를 수행하는 확장형 topic tagging 구조**입니다.

In [0]:
# Databricks notebook source
# COMMAND ----------
# 1) Config

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
ENDPOINT = "databricks-gpt-5-4"

PROMPT_VERSION = "category_topic_dynamic_rules_v1"
SAVE_DB = "sandbox.z_jungryo_lee"

TARGET_CATE_1_DEPTH = "07. 스마트 사용성"
TARGET_CATE_2_DEPTH = "07-06. 리모컨 사용성"
TARGET_SC_MEASUREMENT = 1
RULE_SC_MEASUREMENTS = [1, -1]

RULE_PROFILE_TABLE = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"
TOPIC_POOL_TABLE = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"
DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_{PROMPT_VERSION}"
SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_{PROMPT_VERSION}"

MAX_SAMPLE_ROWS = 1000
MAX_FINAL_TOPICS = 17
MIN_ROWS_PER_TOPIC = 10
CLASSIFY_BATCH_SIZE = 25
MAX_TOKENS = 2200
MAX_RETRIES = 3
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.4

print(
    f"[CONFIG] endpoint={ENDPOINT}, cate_1={TARGET_CATE_1_DEPTH}, cate_2={TARGET_CATE_2_DEPTH}, "
    f"target_sc={TARGET_SC_MEASUREMENT}, max_topics={MAX_FINAL_TOPICS}"
)

In [0]:
# COMMAND ----------
# 2) Helpers

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)

    for candidate in [text]:
        try:
            return json.loads(candidate)
        except Exception:
            pass

    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)

    raise ValueError(f"Invalid JSON: {text[:1000]}")


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def sc_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "긍정"
    if sc_measurement == -1:
        return "부정"
    return "기타"


def overall_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "전반적 긍정"
    if sc_measurement == -1:
        return "전반적 부정"
    return "전반적 평가"


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": max_tokens,
    }

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            print(f"[LLM CALL] attempt={attempt + 1}/{MAX_RETRIES}")
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])
            if isinstance(resp, str):
                return extract_json(resp)
            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            if resp is not None:
                print("[LLM RAW RESPONSE PREVIEW]")
                print(str(resp)[:2000])
            wait_sec = BACKOFF_BASE ** attempt
            print(f"[LLM RETRY WAIT] {wait_sec:.2f}s")
            time.sleep(wait_sec)

    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


def save_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        print(f"[SAVE] overwrite -> {table_name}")
    else:
        print(f"[SAVE] create -> {table_name}")
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def has_specific_reason(text: str, clue_keywords: List[str]) -> bool:
    memo = clean_text(text).lower()
    return any(clean_text(k).lower() in memo for k in clue_keywords if clean_text(k))


def should_be_overall(text: str, clue_keywords: List[str], generic_terms: List[str]) -> bool:
    memo = clean_text(text).lower()
    if not memo:
        return False
    if len(memo) > 20:
        return False
    if has_specific_reason(memo, clue_keywords):
        return False
    return any(clean_text(term).lower() in memo for term in generic_terms if clean_text(term))

In [0]:
# COMMAND ----------
# 3) Load Sample Memos

def load_sample_df(sc_measurement: int, max_rows: int = MAX_SAMPLE_ROWS):
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where 1=1
          and cate_1_depth = '{TARGET_CATE_1_DEPTH}'
          and cate_2_depth = '{TARGET_CATE_2_DEPTH}'
          and sc_measurement = {sc_measurement}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select
            memo,
            row_number() over (
                order by md5(
                    concat(
                        coalesce(memo, ''),
                        '||',
                        '{TARGET_CATE_1_DEPTH}',
                        '||',
                        '{TARGET_CATE_2_DEPTH}',
                        '||',
                        cast({sc_measurement} as string),
                        '||seed_20260420'
                    )
                )
            ) as rn
        from base
    )
    select memo
    from sampled
    where rn <= {max_rows}
    order by rn
    """
    return spark.sql(query).withColumn("_row_id", F.monotonically_increasing_id()).cache()


sample_df = load_sample_df(TARGET_SC_MEASUREMENT, MAX_SAMPLE_ROWS)
print(f"[LOAD] target sample rows = {sample_df.count()}")
display(sample_df.limit(20))


In [0]:
# 4) Generate Dynamic Rule Profiles By sc_measurement

def build_rule_profile_messages(sc_measurement: int, sample_memos: List[str]) -> List[Dict[str, str]]:
    label = sc_label(sc_measurement)
    overall = overall_label(sc_measurement)
    system = f"""
You are a VOC rule designer for TV review topic classification.

Your task:
- Read review memos from one fixed category and one fixed polarity.
- Build reusable rule components for topic classification.

Category:
- cate_1_depth = {TARGET_CATE_1_DEPTH}
- cate_2_depth = {TARGET_CATE_2_DEPTH}
- polarity = {label}

Design outputs:
- overall_topic_label
- clue_keywords: category-specific clue words or short phrases that indicate a specific reason
- generic_terms: very generic sentiment terms that can belong to {overall}
- overall_usage_rule
- specific_reason_rule
- non_overall_examples: examples of memo patterns that must NOT belong to {overall}

Rules:
- clue_keywords must be specialized for this category, not generic for all TV reviews.
- generic_terms must be suitable for this polarity only.
- {overall} should be extremely narrow.
- {overall} must be limited to ultra-short sentiment-only memos where no usable reason can be inferred.
- If a memo mentions usability, design, control, feature, convenience, responsiveness, app use, buttons,
  smart behavior, or any functional impression, it must NOT be treated as {overall}.
- If a memo contains any usable reason, feature, benefit, complaint, or context clue, it must not be treated as {overall}.
- Keep clue_keywords compact and reusable.
- Keep non_overall_examples short.

Return only JSON:
{{
  "overall_topic_label": "",
  "clue_keywords": [""],
  "generic_terms": [""],
  "overall_usage_rule": "",
  "specific_reason_rule": "",
  "non_overall_examples": [""]
}}
"""
    user = "Review memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


rule_rows: List[Dict[str, Any]] = []
for sc in RULE_SC_MEASUREMENTS:
    rule_sample_df = load_sample_df(sc)
    rule_sample_memos = [r["memo"] for r in rule_sample_df.select("memo").toLocalIterator()]
    rule_result = call_llm(build_rule_profile_messages(sc, rule_sample_memos))
    rule_rows.append(
        {
            "cate_1_depth": TARGET_CATE_1_DEPTH,
            "cate_2_depth": TARGET_CATE_2_DEPTH,
            "sc_measurement": sc,
            "overall_topic_label": clean_text(rule_result.get("overall_topic_label")) or overall_label(sc),
            "clue_keywords_json": json.dumps(rule_result.get("clue_keywords", [])[:60], ensure_ascii=False),
            "generic_terms_json": json.dumps(rule_result.get("generic_terms", [])[:30], ensure_ascii=False),
            "overall_usage_rule": clean_text(rule_result.get("overall_usage_rule")),
            "specific_reason_rule": clean_text(rule_result.get("specific_reason_rule")),
            "non_overall_examples_json": json.dumps(rule_result.get("non_overall_examples", [])[:10], ensure_ascii=False),
        }
    )
    rule_sample_df.unpersist()
    time.sleep(RATE_LIMIT_SECONDS)

rule_profile_df = spark.createDataFrame(pd.DataFrame(rule_rows))
save_table(rule_profile_df, RULE_PROFILE_TABLE)
display(rule_profile_df)
# 4) Generate Dynamic Rule Profiles By sc_measurement

def build_rule_profile_messages(sc_measurement: int, sample_memos: List[str]) -> List[Dict[str, str]]:
    label = sc_label(sc_measurement)
    overall = overall_label(sc_measurement)
    system = f"""
You are a VOC rule designer for TV review topic classification.

Your task:
- Read review memos from one fixed category and one fixed polarity.
- Build reusable rule components for topic classification.

Category:
- cate_1_depth = {TARGET_CATE_1_DEPTH}
- cate_2_depth = {TARGET_CATE_2_DEPTH}
- polarity = {label}

Design outputs:
- overall_topic_label
- clue_keywords: category-specific clue words or short phrases that indicate a specific reason
- generic_terms: very generic sentiment terms that can belong to {overall}
- overall_usage_rule
- specific_reason_rule
- non_overall_examples: examples of memo patterns that must NOT belong to {overall}

Rules:
- clue_keywords must be specialized for this category, not generic for all TV reviews.
- generic_terms must be suitable for this polarity only.
- {overall} should be extremely narrow.
- If a memo contains any usable reason, feature, benefit, complaint, or context clue, it must not be treated as {overall}.
- Keep clue_keywords compact and reusable.
- Keep non_overall_examples short.

Return only JSON:
{{
  "overall_topic_label": "",
  "clue_keywords": [""],
  "generic_terms": [""],
  "overall_usage_rule": "",
  "specific_reason_rule": "",
  "non_overall_examples": [""]
}}
"""
    user = "Review memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]

MAX_RULE_SAMPLE_ROWS = 800
rule_rows: List[Dict[str, Any]] = []
for sc in RULE_SC_MEASUREMENTS:
    rule_sample_df = rule_sample_df = load_sample_df(sc, MAX_RULE_SAMPLE_ROWS)
    rule_sample_memos = [r["memo"] for r in rule_sample_df.select("memo").toLocalIterator()]
    rule_result = call_llm(build_rule_profile_messages(sc, rule_sample_memos))
    rule_rows.append(
        {
            "cate_1_depth": TARGET_CATE_1_DEPTH,
            "cate_2_depth": TARGET_CATE_2_DEPTH,
            "sc_measurement": sc,
            "overall_topic_label": clean_text(rule_result.get("overall_topic_label")) or overall_label(sc),
            "clue_keywords_json": json.dumps(rule_result.get("clue_keywords", [])[:60], ensure_ascii=False),
            "generic_terms_json": json.dumps(rule_result.get("generic_terms", [])[:30], ensure_ascii=False),
            "overall_usage_rule": clean_text(rule_result.get("overall_usage_rule")),
            "specific_reason_rule": clean_text(rule_result.get("specific_reason_rule")),
            "non_overall_examples_json": json.dumps(rule_result.get("non_overall_examples", [])[:10], ensure_ascii=False),
        }
    )
    rule_sample_df.unpersist()
    time.sleep(RATE_LIMIT_SECONDS)

rule_profile_df = spark.createDataFrame(pd.DataFrame(rule_rows))
save_table(rule_profile_df, RULE_PROFILE_TABLE)
display(rule_profile_df)


In [0]:
# 5) Load Dynamic Rule Profile For Target sc_measurement

rule_profile_row = (
    rule_profile_df
    .where(F.col("sc_measurement") == F.lit(TARGET_SC_MEASUREMENT))
    .limit(1)
    .collect()[0]
)

OVERALL_TOPIC_LABEL = rule_profile_row["overall_topic_label"]
CLUE_KEYWORDS = json.loads(rule_profile_row["clue_keywords_json"]) if rule_profile_row["clue_keywords_json"] else []
GENERIC_TERMS = json.loads(rule_profile_row["generic_terms_json"]) if rule_profile_row["generic_terms_json"] else []
OVERALL_USAGE_RULE = clean_text(rule_profile_row["overall_usage_rule"])
SPECIFIC_REASON_RULE = clean_text(rule_profile_row["specific_reason_rule"])
NON_OVERALL_EXAMPLES = json.loads(rule_profile_row["non_overall_examples_json"]) if rule_profile_row["non_overall_examples_json"] else []

print("[RULE PROFILE]")
print("overall_topic_label =", OVERALL_TOPIC_LABEL)
print("clue_keywords =", CLUE_KEYWORDS[:20])
print("generic_terms =", GENERIC_TERMS)
print("overall_usage_rule =", OVERALL_USAGE_RULE)
print("specific_reason_rule =", SPECIFIC_REASON_RULE)
print("non_overall_examples =", NON_OVERALL_EXAMPLES)


In [0]:
# 6) Build Topic Pool Using Dynamic Rule Profile

sample_memos = [r["memo"] for r in sample_df.select("memo").toLocalIterator()]

topic_pool_system = f"""
You are a VOC taxonomy designer for TV review topic classification.

Your task:
- Read up to 1000 review memos from one fixed category and polarity.
- Build ONE fixed topic pool that explains WHY users evaluated the category as {sc_label(TARGET_SC_MEASUREMENT)}.

Category:
- cate_1_depth = {TARGET_CATE_1_DEPTH}
- cate_2_depth = {TARGET_CATE_2_DEPTH}

Hard rules:
- Final topic count must be at least 7 and at most {MAX_FINAL_TOPICS}.
- One mandatory topic must be '{OVERALL_TOPIC_LABEL}'.
- {OVERALL_USAGE_RULE}
- {SPECIFIC_REASON_RULE}
- '{OVERALL_TOPIC_LABEL}' must be extremely narrow and only used when no reason can be inferred at all.
- If a memo contains any feature, usability, design, control, convenience, function, response, or app-related clue,
  it must NOT belong to '{OVERALL_TOPIC_LABEL}'.
- Category-specific clue keywords that indicate specific reasons:
  {json.dumps(CLUE_KEYWORDS, ensure_ascii=False)}
- Examples that are NOT '{OVERALL_TOPIC_LABEL}':
  {json.dumps(NON_OVERALL_EXAMPLES, ensure_ascii=False)}
- All non-overall topics should describe a specific reason.
- Topic labels must be Korean.
- topic must be concise noun-like or service/function-like wording.
- description must be a short Korean explanation of what kind of memo belongs there.
- Avoid duplicates and near-synonyms.
- Leave room for one catch-all '기타' topic during post-processing; do not over-fragment topics.

Return only JSON:
{{
  "topics": [
    {{
      "topic": "",
      "description": "",
      "representative_memos": [""]
    }}
  ]
}}
"""

topic_pool_user = "Sample memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])

topic_pool_result = call_llm(
    [
        {"role": "system", "content": clean_text(topic_pool_system)},
        {"role": "user", "content": clean_text(topic_pool_user)},
    ]
)

topic_rows: List[Dict[str, Any]] = []
for order_no, topic in enumerate(topic_pool_result.get("topics", [])[:MAX_FINAL_TOPICS], start=1):
    if not isinstance(topic, dict):
        continue
    topic_rows.append(
        {
            "topic_order": order_no,
            "topic": clean_text(topic.get("topic")),
            "description": clean_text(topic.get("description")),
            "representative_memos_json": json.dumps(topic.get("representative_memos", [])[:5], ensure_ascii=False),
        }
    )

topic_pool_df = spark.createDataFrame(pd.DataFrame(topic_rows))
save_table(topic_pool_df, TOPIC_POOL_TABLE)
display(topic_pool_df)

In [0]:
# 7) Classify Using Dynamic Rule Profile

topic_pool_payload = [
    {
        "topic": r["topic"],
        "description": r["description"],
    }
    for r in topic_pool_df.orderBy("topic_order").toLocalIterator()
]


def build_classify_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    system = f"""
You are a VOC classifier for TV review topic classification.

Classify each memo into exactly one topic from the fixed topic list.
Every memo must belong to exactly one topic.

Rules:
- The task is to identify WHY the writer evaluated {TARGET_CATE_2_DEPTH} as {sc_label(TARGET_SC_MEASUREMENT)}.
- Overall topic is '{OVERALL_TOPIC_LABEL}'.
- {OVERALL_USAGE_RULE}
- {SPECIFIC_REASON_RULE}
- '{OVERALL_TOPIC_LABEL}' is allowed only for ultra-short sentiment-only memos with no usable reason.
- If a memo contains any specific reason, function, usability signal, design signal, or feature impression,
  it must be assigned to a non-overall topic.
- clue keywords indicating specific reasons:
  {json.dumps(CLUE_KEYWORDS, ensure_ascii=False)}
- examples that are NOT '{OVERALL_TOPIC_LABEL}':
  {json.dumps(NON_OVERALL_EXAMPLES, ensure_ascii=False)}
- Do not invent any new topic.
- For multilingual memos, understand them and reason in Korean.
- explanation must be a short Korean sentence explaining why the memo belongs to the topic.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Fixed topics:\n"
        + json.dumps(topic_pool_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


def build_reclassify_without_overall_message(row: Dict[str, Any]) -> List[Dict[str, str]]:
    specific_topic_payload = [topic for topic in topic_pool_payload if clean_text(topic["topic"]) != OVERALL_TOPIC_LABEL]
    system = f"""
You are a VOC classifier for TV review topic classification.

This memo was incorrectly over-generalized as '{OVERALL_TOPIC_LABEL}'.
Reclassify it using only the non-general topics below.

Rules:
- Choose exactly one non-general topic.
- The memo contains a specific reason and must not remain as '{OVERALL_TOPIC_LABEL}'.
- Use the clue keywords below as hints for specific reasons:
  {json.dumps(CLUE_KEYWORDS, ensure_ascii=False)}
- explanation must be a short Korean sentence explaining the specific reason.

Return only JSON:
{{
  "topic": "",
  "explanation": ""
}}
"""
    user = (
        "Allowed non-general topics:\n"
        + json.dumps(specific_topic_payload, ensure_ascii=False)
        + "\n\nMemo:\n"
        + json.dumps({"memo": clean_text(row["memo"])}, ensure_ascii=False)
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


classified_rows: List[Dict[str, Any]] = []
source_rows = [r.asDict(recursive=True) for r in sample_df.toLocalIterator()]

for batch in chunk_list(source_rows, CLASSIFY_BATCH_SIZE):
    batch_result = call_llm(build_classify_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        topic_raw = clean_text(mapped.get("topic"))
        explanation_raw = clean_text(mapped.get("explanation"))

        if (
            topic_raw == OVERALL_TOPIC_LABEL
            and has_specific_reason(row["memo"], CLUE_KEYWORDS)
            and not should_be_overall(row["memo"], CLUE_KEYWORDS, GENERIC_TERMS)
        ):
            retry_result = call_llm(build_reclassify_without_overall_message(row))
            retry_topic = clean_text(retry_result.get("topic"))
            retry_explanation = clean_text(retry_result.get("explanation"))
            if retry_topic:
                topic_raw = retry_topic
            if retry_explanation:
                explanation_raw = retry_explanation

        classified_rows.append(
            {
                "_row_id": row["_row_id"],
                "memo": row["memo"],
                "topic_raw": topic_raw,
                "explanation_raw": explanation_raw,
            }
        )
    time.sleep(RATE_LIMIT_SECONDS)

classified_df = spark.createDataFrame(pd.DataFrame(classified_rows))
display(classified_df.limit(20))


In [0]:
# 8) Merge Small Topics (<10) Into '기타'

topic_stats_df = classified_df.groupBy("topic_raw").agg(F.count("*").alias("topic_cnt"))
rare_topics = [r["topic_raw"] for r in topic_stats_df.where(F.col("topic_cnt") < F.lit(MIN_ROWS_PER_TOPIC)).collect()]
rare_topics_sorted = sorted([t for t in rare_topics if clean_text(t)])
rare_topic_label = f"기타({', '.join(rare_topics_sorted[:3])})" if rare_topics_sorted else "기타"

topic_desc_map = {
    r["topic"]: r["description"]
    for r in topic_pool_df.select("topic", "description").toLocalIterator()
}

final_rows: List[Dict[str, Any]] = []
for row in classified_df.toLocalIterator():
    row_dict = row.asDict(recursive=True)
    raw_topic = clean_text(row_dict["topic_raw"])
    raw_expl = clean_text(row_dict["explanation_raw"])

    if raw_topic in rare_topics_sorted:
        final_topic = rare_topic_label
        final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
    else:
        final_topic = raw_topic if raw_topic else "기타"
        final_explanation = raw_expl or topic_desc_map.get(final_topic, "")

    final_rows.append(
        {
            "_row_id": row_dict["_row_id"],
            "memo": row_dict["memo"],
            "topic": final_topic,
            "description": final_explanation,
        }
    )

detail_df = spark.createDataFrame(pd.DataFrame(final_rows))
display(detail_df.limit(20))


In [0]:
# COMMAND ----------
# 9) Load Full Population And Exclude Already-Classified Sample Rows

full_query = f"""
with base as (
    select distinct
        memo
    from {SOURCE_TABLE}
    where 1=1
      and cate_1_depth = '{TARGET_CATE_1_DEPTH}'
      and cate_2_depth = '{TARGET_CATE_2_DEPTH}'
      and sc_measurement = {TARGET_SC_MEASUREMENT}
      and memo is not null
      and length(trim(memo)) > 0
)
select memo
from base
"""

full_df = (
    spark.sql(full_query)
    .withColumn("_row_id", F.monotonically_increasing_id())
    .withColumn("row_key", F.md5(F.concat_ws("||", F.lit(TARGET_CATE_1_DEPTH), F.lit(TARGET_CATE_2_DEPTH), F.lit(str(TARGET_SC_MEASUREMENT)), F.col("memo"))))
    .cache()
)

sample_key_df = (
    sample_df
    .withColumn("row_key", F.md5(F.concat_ws("||", F.lit(TARGET_CATE_1_DEPTH), F.lit(TARGET_CATE_2_DEPTH), F.lit(str(TARGET_SC_MEASUREMENT)), F.col("memo"))))
    .select("row_key")
    .distinct()
)

remaining_df = (
    full_df.join(sample_key_df, on="row_key", how="left_anti")
    .drop("row_key")
    .cache()
)

print(f"[FULL] total rows = {full_df.count()}")
print(f"[SAMPLE] already classified sample rows = {sample_key_df.count()}")
print(f"[REMAINING] rows to classify = {remaining_df.count()}")

display(remaining_df.limit(20))


In [0]:
# COMMAND ----------
# 10) Classify Remaining Rows With The Same Topic Pool / Rules

remaining_rows = [r.asDict(recursive=True) for r in remaining_df.toLocalIterator()]
remaining_classified_rows: List[Dict[str, Any]] = []

for batch in chunk_list(remaining_rows, CLASSIFY_BATCH_SIZE):
    batch_result = call_llm(build_classify_messages(batch))
    result_map = {
        str(item.get("row_id")): item
        for item in batch_result.get("results", [])
        if isinstance(item, dict)
    }

    for row in batch:
        mapped = result_map.get(str(row["_row_id"]), {})
        topic_raw = clean_text(mapped.get("topic"))
        explanation_raw = clean_text(mapped.get("explanation"))

        if (
            topic_raw == OVERALL_TOPIC_LABEL
            and has_specific_reason(row["memo"], CLUE_KEYWORDS)
            and not should_be_overall(row["memo"], CLUE_KEYWORDS, GENERIC_TERMS)
        ):
            retry_result = call_llm(build_reclassify_without_overall_message(row))
            retry_topic = clean_text(retry_result.get("topic"))
            retry_explanation = clean_text(retry_result.get("explanation"))
            if retry_topic:
                topic_raw = retry_topic
            if retry_explanation:
                explanation_raw = retry_explanation

        remaining_classified_rows.append(
            {
                "_row_id": row["_row_id"],
                "memo": row["memo"],
                "topic_raw": topic_raw,
                "explanation_raw": explanation_raw,
            }
        )

    time.sleep(RATE_LIMIT_SECONDS)

remaining_classified_df = spark.createDataFrame(pd.DataFrame(remaining_classified_rows))
display(remaining_classified_df.limit(20))


In [0]:
# COMMAND ----------
# 11) Combine Sample Classification + Remaining Classification

combined_classified_df = classified_df.unionByName(remaining_classified_df)

print(f"[COMBINED] classified rows = {combined_classified_df.count()}")
display(combined_classified_df.limit(20))


In [0]:
# COMMAND ----------
# 12) Rebuild Final Topic Merge On Combined Data

combined_topic_stats_df = (
    combined_classified_df.groupBy("topic_raw")
    .agg(F.count("*").alias("topic_cnt"))
)

combined_rare_topics = [
    r["topic_raw"]
    for r in combined_topic_stats_df.where(F.col("topic_cnt") < F.lit(MIN_ROWS_PER_TOPIC)).collect()
]
combined_rare_topics_sorted = sorted([t for t in combined_rare_topics if clean_text(t)])
combined_rare_topic_label = f"기타({', '.join(combined_rare_topics_sorted[:3])})" if combined_rare_topics_sorted else "기타"

topic_desc_map = {
    r["topic"]: r["description"]
    for r in topic_pool_df.select("topic", "description").toLocalIterator()
}

combined_final_rows: List[Dict[str, Any]] = []
for row in combined_classified_df.toLocalIterator():
    row_dict = row.asDict(recursive=True)
    raw_topic = clean_text(row_dict["topic_raw"])
    raw_expl = clean_text(row_dict["explanation_raw"])

    if raw_topic in combined_rare_topics_sorted:
        final_topic = combined_rare_topic_label
        final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
    else:
        final_topic = raw_topic if raw_topic else "기타"
        final_explanation = raw_expl or topic_desc_map.get(final_topic, "")

    combined_final_rows.append(
        {
            "_row_id": row_dict["_row_id"],
            "memo": row_dict["memo"],
            "topic": final_topic,
            "description": final_explanation,
        }
    )

combined_detail_df = spark.createDataFrame(pd.DataFrame(combined_final_rows))
display(combined_detail_df.limit(20))

In [0]:
# COMMAND ----------
# 13) Final Summary On Full Population

combined_summary_df = (
    combined_detail_df.groupBy("topic")
    .agg(F.count("*").alias("review_count"))
    .withColumn("total_count", F.sum("review_count").over(Window.partitionBy()))
    .withColumn("review_share", F.round(F.col("review_count") / F.col("total_count"), 4))
    .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
    .orderBy(F.desc("review_count"), "topic")
)

display(combined_summary_df)


In [0]:
# COMMAND ----------
# 14) Save Final Full Result Tables

FULL_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_full_{PROMPT_VERSION}"
FULL_SUMMARY_TABLE = f"{SAVE_DB}.category_topic_summary_full_{PROMPT_VERSION}"

save_table(combined_detail_df, FULL_DETAIL_TABLE)
save_table(combined_summary_df, FULL_SUMMARY_TABLE)

print(f"""
-- Full Detail
select *
from {FULL_DETAIL_TABLE}
order by topic, _row_id;

-- Full Summary
select *
from {FULL_SUMMARY_TABLE}
order by review_count desc, topic;
""")
